In [1]:
from misc import *
import constant

# upto 5 long track candidates isMuon
LTRACK5_FILE_PATH = "/dice/projects/LHCb/PbPb_MSCi/isMuon_Tr1_5_B2JpsiK_PbPb/LeadLead_B2JpsiKTuple_all.root"
DEFAULT_TREE_NAME = "B2JpsiKTuple/DecayTree"

df = load_root(LTRACK5_FILE_PATH, DEFAULT_TREE_NAME)
df = df[df["Kplus_isMuon"] == False]
df = df[(constant.JPSI_MASS-50 <= df["J_psi_1S_M"]) & (df["J_psi_1S_M"] <= constant.JPSI_MASS+50)]

In [10]:
import zfit
from zfit.loss import ExtendedUnbinnedNLL
from zfit.minimize import Minuit
import numpy as np

bounds = (0.1, 3.0)
obs = zfit.Space('x', limits=bounds)
bkg = np.random.exponential(1/2, 300)
peak = np.random.normal(1.2, 0.1, 10)
data = np.concatenate((bkg, peak))
data = data[(data > bounds[0]) & (data < bounds[1])]
data = zfit.Data.from_numpy(obs=obs, array=data)

lambda_ = zfit.Parameter("lambda", -2.0, -4.0, -1.0)
Nsig = zfit.Parameter("Nsig", 1., -20., 500)
Nbkg = zfit.Parameter("Nbkg", 250, 0., 500)
signal = zfit.pdf.Gauss(obs=obs, mu=1.2, sigma=0.1).create_extended(Nsig)
background = zfit.pdf.Exponential(obs=obs, lambda_=lambda_).create_extended(Nbkg)
total = zfit.pdf.SumPDF([signal, background])
nll = ExtendedUnbinnedNLL(model=total, data=data)

# Instantiate a minuit minimizer
minimizer = Minuit()
# minimisation of the loss function
minimum = minimizer.minimize(loss=nll)
minimum.hesse()
print(minimum)

from hepstats.hypotests.calculators import AsymptoticCalculator
calculator = AsymptoticCalculator(nll, Minuit(), asimov_bins=100)

from hepstats.hypotests.parameters import POI, POIarray
# background only
poialt = POI(Nsig, 0)
# background + signal
poinull = POIarray(Nsig, np.linspace(0.0, 25, 20))

from hepstats.hypotests import UpperLimit
ul = UpperLimit(calculator, poinull, poialt)
ul.upperlimit(alpha=0.05, CLs=True)

FitResult of
<ExtendedUnbinnedNLL model=[<zfit.<class 'zfit.models.functor.SumPDF'>  params=[Composed_autoparam_6, Composed_autoparam_7]] data=[<zfit.Data: Data obs=('x',) shape=(259, 1)>] constraints=[]> 
with
<Minuit Minuit tol=0.001>

╒═════════╤═════════════╤══════════════════╤════════╤══════════════════════════════╕
│  valid  │  converged  │  param at limit  │  edm   │   approx. fmin (full | opt.) │
╞═════════╪═════════════╪══════════════════╪════════╪══════════════════════════════╡
│  True   │    True     │      False       │ 0.0002 │             95.61 |  10041.6 │
╘═════════╧═════════════╧══════════════════╧════════╧══════════════════════════════╛

Parameters
name      value  (rounded)        hesse    at limit
------  ------------------  -----------  ----------
Nsig               7.01969  +/-     6.2       False
Nbkg               250.947  +/-     4.8       False
lambda            -1.91007  +/-    0.14       False


{'observed': 18.899016340271466,
 'expected': 11.511036107693755,
 'expected_p1': 16.793213599481568,
 'expected_m1': 8.012023041934505,
 'expected_p2': 23.80281331329218,
 'expected_m2': 5.832612326788863}